### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ljubljana_breast_cancer",
    dataset_year="1988",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51P4M",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/14/breast+cancer.zip && unzip breast+cancer.zip breast-cancer.data && rm breast+cancer.zip && mkdir -p local-data-warehouse/ljubljana_breast_cancer && mv breast-cancer.data local-data-warehouse/ljubljana_breast_cancer/
""",
    # References
    academic_reference_bibtex="""@misc{Zwitter1988BreastCancer,
  author       = {Zwitter, Matjaz and Soklic, Milan},
  title        = {{Breast Cancer}},
  year         = {1988},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C51P4M}
}
""",
    academic_reference_bibtex_key="Zwitter1988BreastCancer",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We encode missing values as np.nan instead of "?".
- We reverse the discretization of several numeric features and make them integers again.
- We keep duplicates as they seem naturally occurring.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Class",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import re

columns = [
    "Class",
    "age",
    "menopause",
    "tumor-size",
    "inv-nodes",
    "node-caps",
    "deg-malig",
    "breast",
    "breast-quad",
    "irradiat"
]
df = pd.read_csv(dataset_mold.path / "breast-cancer.data", header=None, names=columns, na_values="?")
print("Loaded data shape:", df.shape)


def interval_midpoint(x):
    """
    Convert strings like '10-19' -> 14 (mean of boundaries, rounded),
    '?' / NaN -> NaN, already-numeric -> as-is.
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return x

    s = str(x).strip()
    if s in {"?", ""}:
        return np.nan

    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", s)
    if not m:
        return np.nan  # or return s if you prefer to keep non-interval values

    a = int(m.group(1))
    b = int(m.group(2))
    return (a + b) / 2

for c in ["age", "tumor-size", "inv-nodes"]:
    df[c] = df[c].map(interval_midpoint)

as_cat_type = ["Class", "menopause", "node-caps", "breast", "breast-quad", "irradiat"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()